<a href="https://colab.research.google.com/github/TharunThirumaran/Deep-learning/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets

In [ ]:
import re
import numpy as np
import pandas as pd

from datasets import load_dataset

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

from sklearn.model_selection import train_test_split

In [ ]:
dataset = load_dataset("wangrongsheng/ag_news")
# gets only training , testing data
train_data = dataset["train"]
test_data = dataset["test"]

In [ ]:
# converts the hugging face dataset into pandas dataframe
train_df = train_data.to_pandas()
test_df = test_data.to_pandas()

print(train_df.head())

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2


# Welcome to Colab!

In [ ]:
def clean_text(text):
  #converts everything to lowercase
    text = text.lower()
  #it replaces the matching characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text
#applies the clean text function to every row in the text clean row to the text column.
train_df["text"] = train_df["text"].apply(clean_text)
test_df["text"] = test_df["text"].apply(clean_text)

In [ ]:
#stores the news articles
X_train = train_df["text"]
y_train = train_df["label"]
#stores the class label
X_test = test_df["text"]
y_test = test_df["label"]

In [ ]:
max_words = 10000

tokenizer = Tokenizer(num_words=max_words)
#builds the vocabulary from the training text
tokenizer.fit_on_texts(X_train)
#converts each article into a sequence of integers
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [ ]:
#each article will be represented by 50 words
max_length = 50

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

In [ ]:
#creates the neural networks
model = Sequential()

model.add(
    Embedding(
        input_dim=max_words,
        output_dim=128,
        input_length=max_length
    )
)

model.add(SimpleRNN(64))

model.add(Dense(4, activation='softmax'))

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 60s 38ms/step - accuracy: 0.7888 - loss: 0.5802 - val_accuracy: 0.8337 - val_loss: 0.4543
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 54s 36ms/step - accuracy: 0.8812 - loss: 0.3613 - val_accuracy: 0.8693 - val_loss: 0.4110
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 81s 36ms/step - accuracy: 0.9069 - loss: 0.2909 - val_accuracy: 0.8688 - val_loss: 0.3905
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 83s 36ms/step - accuracy: 0.9208 - loss: 0.2460 - val_accuracy: 0.8732 - val_loss: 0.4169
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 82s 36ms/step - accuracy: 0.9320 - loss: 0.2108 - val_accuracy: 0.8783 - val_loss: 0.3880


In [ ]:
loss, accuracy = model.evaluate(X_test_pad, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8833 - loss: 0.3837
Test Loss: 0.3836570382118225
Test Accuracy: 0.8832894563674927


In [ ]:
news = [
    "India defeated Australia by six wickets in the cricket world cup."
]

news = [clean_text(x) for x in news]

sequence = tokenizer.texts_to_sequences(news)

padded = pad_sequences(
    sequence,
    maxlen=max_length,
    padding='post'
)

prediction = model.predict(padded)

predicted_class = np.argmax(prediction)

labels = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

print("Predicted Category:", labels[predicted_class])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 299ms/step
Predicted Category: Sports
